# Forward Problem

To solve the forward problem, one must define a head model. The head model includes:
1. The brain, represented by its arbitrarly defined brain sources.
2. The space between the sources and the scalp.
3. The position of the electrodes on the scalp.

Thanks to FreeSurfer, we already computed the brain surface. So we can start from there and the T1-weighted image of the subject.

## 📦 Imports

In [1]:
import mindmap as mm
import mne
from pathlib import Path
import pyvista as pv
import nibabel as nib
import numpy as np

## 🧠 Data

For this section, we mainly need the T1-weighted MRI of the subject (we don't explicitly download this file because Python-MNE knows where each required file is located within the FreeSurfer directory) and the electrode positions. The parcellation is used to obtain a gray matter mask, which will help constrain the location of the sources.

In [3]:
sub     = "sub-pat10"
dataset = "spike"
scale   = 4

# configuration
config = mm.io.read_config("../configurations/spike.yml")
fs_dir = Path(config["FS_DIR"])
mm_dir = Path(config["MM_DIR"])

# files
mri  = fs_dir/sub/"mri"/"T1.mgz"                                                                        # for visualization
eeg  = f"/home/nino/Data/eeg_epi_spikeerp_data_1.0.0/derivatives/cartool-v3.80/{sub}/ses-preop/eeg"
parc = mm_dir/sub/"parc"/f"{sub}_multiConn{scale}-2-0.05_dparc.nii.gz"                                  # for gray matter mask
clut = mm_dir/sub/"parc"/f"{sub}_multiConn{scale}-2-0.05_clut.txt"

# load
mri  = nib.load(mri)
eeg  = mm.io.read_eeg(eeg, dataset)
parc = mm.utils.Parcellation(parc, clut)

df   = parc.get_clut()
ctx  = df[df["Label name"].str.startswith("ctx")]
mask = parc.create_mask(ctx["#No."].values)


# parameters
constraint = {"type":"surf", "mask":mask}
spacing    = 7
n_jobs     = 10
conductivities=[0.3, 0.006, 0.3]
min_skull_thick=2

# plotting parameters
plot_bem_kwargs = dict(
    subject=sub,
    subjects_dir=fs_dir,
    brain_surfaces="white",
    orientation="coronal",
    slices=[50, 100, 150, 200]
)

## Source Space

#### ⚙️ <font color=green>Define sources along the brain surface</font>

We distribute the sources uniformly along the brain surface (more specifically, the white surface— the boundary between gray and white matter). The number of sources is adjustable based on the `spacing`. We then eliminate any source that is not located within the gray matter, as only the synaptic currents occurring in the gray matter are believed to contribute to EEG.

In [4]:
# surface-based
if constraint["type"] == "surf": # (1 min)
    sources = mne.setup_source_space(
        subject=sub,
        spacing=spacing,
        surface="white",
        add_dist=True,      # geodesic distances between sources
        subjects_dir=fs_dir,
        n_jobs=n_jobs,
        verbose="warning",
    )

# volumetric
elif constraint["type"] == "vol":
    sources = mne.setup_volume_source_space(
        subject=sub,
        pos=spacing,
        subjects_dir=fs_dir,
        verbose="warning"
    )
    
# gray matter constraint 
if constraint["mask"] is not None:
    sources = mm.utils.mask_sources(sources, constraint["mask"], mask.affine)

print("\033[92mSUMMARY\033[0m")
print(f"Gray matter constraint: {constraint['mask'] is not None}\n")
for src in sources:
    print(f"Source space id: {src["id"]}")
    print(f"Type: {src["type"]}")
    print(f"Number of sources: {src["nuse"]}\n")

SUMMARY
Gray matter constraint: True

Source space id: 101 (FIFFV_MNE_SURF_LEFT_HEMI)
Type: surf
Number of sources: 3211

Source space id: 102 (FIFFV_MNE_SURF_RIGHT_HEMI)
Type: surf
Number of sources: 3161



## Inter source-electrode space

The space between the sources and the electrodes is modeled here using a Boundary Element Model (BEM). This model represents the boundaries between the sources and the target, corresponding to the inner skull, outer skull, and outer skin. The space between these boundaries is considered homogeneous for simplicity.

#### ⚙️ <font color=green>Create surfaces (a.k.a boundaries) from T1-weighted MRI using FreeSurfer's implemented Watershed</font>

In [10]:
mne.bem.make_watershed_bem(
    subject = sub,
    subjects_dir = fs_dir,
    volume = "T1",
    overwrite = True,
    verbose = "warning",
)

#### <font color=yellow>🌟</font> Visualize surfaces

In [4]:
names    = ["outer_skin.surf", "outer_skull.surf", "inner_skull.surf"]
images   = []
for n in names:
    rr, tri = nib.freesurfer.read_geometry(fs_dir/sub/"bem"/n)
    rr = nib.affines.apply_affine(np.linalg.inv(mri.affine), rr).astype(int)

    vol = np.zeros(mri.shape)
    vol[rr[:,0], rr[:,1], rr[:,2]] = 255

    img = pv.ImageData(dimensions=mri.shape, spacing=(1,1,1), origin=(0,0,0))
    img.point_data["scalars"] = vol.flatten(order="F")

    images.append(img)


volume = pv.ImageData(dimensions=mri.shape, spacing=(1,1,1), origin=(0,0,0))
volume.point_data["scalars"] = mri.get_fdata().flatten(order="F")

plotter = pv.Plotter()
plotter.add_mesh(volume.contour(3), color="white", opacity=0.05)

def create_slice(normal, origin):
    slc_mri    = volume.slice(normal=normal, origin=origin)
    slc_skin   = images[0].slice(normal=normal, origin=origin)
    slc_oskull = images[1].slice(normal=normal, origin=origin)
    slc_iskull = images[2].slice(normal=normal, origin=origin)
    plotter.add_mesh(slc_mri, cmap="bone", name="mri", opacity="linear")
    plotter.add_mesh(slc_skin, cmap="Reds", name="skin", opacity="linear")
    plotter.add_mesh(slc_oskull, cmap="Reds", name="oskull", opacity="linear")
    plotter.add_mesh(slc_iskull, cmap="Greens", name="iskull", opacity="linear")


plotter.add_plane_widget(
    create_slice,
)
plotter.show_grid()
plotter.show_axes()
plotter.show()

def plot_slices(nifti, surfaces):
    pass

Widget(value='<iframe src="http://localhost:40649/index.html?ui=P_0x7f7cb8499d30_0&reconnect=auto" class="pyvi…

#### ⚙️ <font color=green>Correct surfaces</font>
The watershed algorithm sometimes generates an inner skull that extends beyond the outer skull. We have implemented some functions to correct this issue.

In [11]:
from mindmap.utils.mri import correct_skull 

correct_skull(
    sub,
    offset = 2,
    fs_dir = fs_dir,
    save_arrows = True,
)

# Correct skull sends an "ERROR:root:No points/cells to operate on" when there are no 
# correction to be made. The program will continue without stopping.

ERROR:root:No points/cells to operate on
2024-10-18 10:26:47.953 ( 452.269s) [    7FA3590F6740]vtkIntersectionPolyData:2423  WARN| No Intersection between objects 
2024-10-18 10:26:47.953 ( 452.270s) [    7FA3590F6740]vtkDistancePolyDataFilt:73     ERR| vtkDistancePolyDataFilter (0x55623c6b5fa0): No points/cells to operate on
ERROR:root:No points/cells to operate on
2024-10-18 10:26:47.954 ( 452.271s) [    7FA3590F6740]vtkDistancePolyDataFilt:73     ERR| vtkDistancePolyDataFilter (0x55623c6b5fa0): No points/cells to operate on


#### ⚙️ <font color=green>Assemble Boudary Element Model (BEM)</font>
In this section, we apply topological checks to verify if the surfaces are compatible with each other in order to create a conductor model later.

In [12]:
head_model = mne.make_bem_model(
        subject=sub,
        subjects_dir=fs_dir,
        conductivity=conductivities,
        #ico=None,
        verbose="warning",
)

## Register Electrodes
For the forward solution to be accurate, the electrodes must be aligned with the scalp.

#### <font color=yellow>🌟</font> Visualize alignmnent

In [13]:
points = mm.utils.get_ch_positions(eeg)
elec   = pv.PolyData(points)
head   = mm.utils.surf_to_polydata(head_model[0]["rr"], head_model[0]["tris"])

plotter = pv.Plotter()
plotter.add_mesh(elec, render_points_as_spheres=True, color="red", name="initial electrodes")
plotter.add_mesh(head, smooth_shading=True, color="lightgray", name="head")
plotter.show()

Widget(value='<iframe src="http://localhost:32997/index.html?ui=P_0x7fa1eb2a3500_0&reconnect=auto" class="pyvi…

⚙️ <font color=green>**Iterative Closest Point (ICP)**</font>: ICP is an algorithm employed to minimize the difference between two clouds of points.

In [ ]:
#new_points       = mne.transforms.apply_trans(trans, elec)
#new_points, _, _ = mm.utils.point_to_plane_icp_with_scaling(new_points, head, max_iterations=1)
#_, _, new_points = mm.utils.optimize_scaling_translation(new_points, head)

⚙️ <font color=green>**Projection to Closest Neightbor**</font>: To ensure the electrodes are actually on the scalp, we project them to the nearest point on the scalp.

In [14]:
points = mm.utils.project_points_to_surface(points, head)

In [15]:
elec_new = pv.PolyData(points)
_ = plotter.add_mesh(elec_new, render_points_as_spheres=True, color="green", name="new electrodes")

#### 🔄 Update new electrode positions

In [16]:
data     = dict(zip(eeg.info["ch_names"], points))
montage  = mne.channels.make_dig_montage(ch_pos=data, coord_frame="head")
eeg      = eeg.set_montage(montage)

## Solve Forward Problem

⚙️ <font color=green> **Solve the conduction problem and create the Leadfield matrix**</font>: Now that the head model is complete, we can compute the lead field matrix, which describes the projection of the sources onto the electrodes. To do this, we solve the conductivity problem, treating the head model as the space.

In [18]:
bem = mne.make_bem_solution(head_model)
trans = mne.transforms.Transform(fro="head", to="mri")      # identity matrix

forward = mne.make_forward_solution(
    info=eeg.info,
    trans=trans,
    src=sources,
    bem=bem,
    meg=False,
    eeg=True,
    verbose="warning",
)

Three-layer model surfaces loaded.
Computing the linear collocation solution...
    Matrix coefficients...
        outer skin  (2562) -> outer skin  (2562) ...
        outer skin  (2562) -> outer skull (2562) ...
        outer skin  (2562) -> inner skull (2562) ...
        outer skull (2562) -> outer skin  (2562) ...
        outer skull (2562) -> outer skull (2562) ...
        outer skull (2562) -> inner skull (2562) ...
        inner skull (2562) -> outer skin  (2562) ...
        inner skull (2562) -> outer skull (2562) ...
        inner skull (2562) -> inner skull (2562) ...
    Inverting the coefficient matrix...
IP approach required...
    Matrix coefficients (homog)...
        inner skull (2562) -> inner skull (2562) ...
    Inverting the coefficient matrix (homog)...
    Modify the original solution to incorporate IP approach...
        Combining...
        Scaling...
Solution ready.
BEM geometry computations complete.


#### <font color=yellow>🌟</font> Visualize Forward solution

In [19]:
mm.viz.plot_leadfield(eeg, forward).show()

    Average patch normals will be employed in the rotation to the local surface coordinates....
    Converting to surface-based source orientations...
    [done]


Widget(value='<iframe src="http://localhost:32997/index.html?ui=P_0x7fa1c29a3440_1&reconnect=auto" class="pyvi…

## 💾 Save Forward Solution

In [21]:
# save source space
folder_fwd = mm_dir/sub/"fwd"
folder_eeg = mm_dir/sub/"eeg"

if folder_fwd.exists() == False:
        folder_fwd.mkdir(parents=True)
if folder_eeg.exists() == False:
        folder_eeg.mkdir(parents=True)

print(f"Saving head model and forward solution for subject: {sub}")
mne.write_bem_surfaces(folder_fwd/f"{sub}_hm-bem.fif", head_model, overwrite=True)
mne.write_forward_solution(folder_fwd/f"{sub}_bem-fwd.fif", forward, overwrite=True)

print(f"Saving co-registered eeg for subject: {sub}")
if dataset == "spike":
        eeg.save(folder_eeg/f"{sub}_coreg_ave.fif", overwrite=True)
else:
        raise ValueError(f"Unknown dataset: {dataset}")

Saving head model and forward solution for subject: sub-pat10
Overwriting existing file.
Overwriting existing file.
    Write a source space...
    [done]
    Write a source space...
    [done]
    2 source spaces written
Saving co-registered eeg for subject: sub-pat10
Overwriting existing file.
